# Session 2 · Configuring Hadoop and Setting Up a Cluster

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Diploma-in-Data-Science/blob/main/term-2/02-big-data-systems-with-hadoop/code/session-02-configuring-hadoop-cluster.ipynb)

Diploma in Data Science · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run the cells from top to bottom. Every example is self-contained, so you can also run any single cell on its own.

### Example 3 · Reading configuration like Hadoop does

In [1]:
import xml.etree.ElementTree as ET

defaults = {"fs.defaultFS": "file:///", "dfs.replication": "3", "dfs.blocksize": "134217728",
            "mapreduce.framework.name": "local", "yarn.nodemanager.resource.memory-mb": "8192"}

site_xml = {
 "core-site.xml": """<configuration><property><name>fs.defaultFS</name>
     <value>hdfs://localhost:9000</value></property></configuration>""",
 "hdfs-site.xml": """<configuration><property><name>dfs.replication</name><value>1</value></property>
     </configuration>""",
 "mapred-site.xml": """<configuration><property><name>mapreduce.framework.name</name>
     <value>yarn</value></property></configuration>""",
}

effective = dict(defaults)
source = {k: "default" for k in defaults}
for fname, text in site_xml.items():
    for prop in ET.fromstring(text).findall("property"):
        name, value = prop.findtext("name").strip(), prop.findtext("value").strip()
        effective[name], source[name] = value, fname

for k in effective:
    print(f"{k:<38} = {effective[k]:<24} ({source[k]})")
print("\nblock size in MB:", int(effective["dfs.blocksize"]) // (1024 * 1024))

fs.defaultFS                           = hdfs://localhost:9000    (core-site.xml)
dfs.replication                        = 1                        (hdfs-site.xml)
dfs.blocksize                          = 134217728                (default)
mapreduce.framework.name               = yarn                     (mapred-site.xml)
yarn.nodemanager.resource.memory-mb    = 8192                     (default)

block size in MB: 128


### Example 4 · Sizing a cluster

In [2]:
import math
today_tb, growth_tb_month, months = 50, 1.5, 24
replication, temp_overhead, max_fill = 3, 0.25, 0.80
disks_per_node, disk_tb = 12, 8

data = today_tb + growth_tb_month * months
raw = data * replication * (1 + temp_overhead) / max_fill
node_tb = disks_per_node * disk_tb
nodes = math.ceil(raw / node_tb)

print(f"data after {months} months     : {data:.1f} TB")
print(f"raw capacity needed       : {raw:.1f} TB")
print(f"capacity per worker node  : {node_tb} TB ({disks_per_node} x {disk_tb} TB)")
print(f"worker nodes needed       : {nodes}")
print(f"tolerating 2 failed nodes : {nodes + 2}")

data after 24 months     : 86.0 TB
raw capacity needed       : 403.1 TB
capacity per worker node  : 96 TB (12 x 8 TB)
worker nodes needed       : 5
tolerating 2 failed nodes : 7
